In [ ]:
import numpy
import pandas
import time
import tensorflow as tf
import random
import os

from keras import optimizers
from keras.utils import plot_model
from keras.layers import Dense, LSTM ,Dropout, SimpleRNN, Conv1D, MaxPooling1D, Input
from keras_model import ATSLSTM, AttentionLayer
from keras.models import Sequential, load_model
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
import joblib
from tqdm import trange
from math import sqrt

In [2]:
def set_seed(seed=42):
    """
    Set seed for reproducibility
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.random.set_seed(seed)
    numpy.random.seed(seed)
    random.seed(seed)

set_seed()

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [ ]:
# Optimized parameters
# Starting Point (SP) có 2 ý nghĩa:
# 1. Số lượng cycles trong quá khứ của mỗi pin được phép sử dụng để training
# 2. Thời điểm bắt đầu dự đoán RUL (dự đoán từ (SP + 1) cycle trở đi)
# 3. SP khác LOOK_BACK
# 1.1e-2


BLOCKS_NUM = 84
LR = 2.6e-3
BATCH_SIZE = 17
EPOCHS = 170
DROPOUT_RATE = 1.1e-2

# BLOCKS_NUM = 64
# LR = 0.001
# BATCH_SIZE = 32
# EPOCHS = 100
# DROPOUT_RATE = 0.01

SP = 50
LOOK_BACK = 30
SAMPLE_NUM = SP - LOOK_BACK
TIMESTEPS = 168 - SP
TRUE_RUL = 124 - SP

In [4]:
def load_dataset(datasource1: str, datasource2: str, datasource3: str, datasource4: str, sp: int) -> (numpy.ndarray, MinMaxScaler):
    # SP cycles of the first 3 batteries and whole cycles of the last battery are used
    dataframe1 = pandas.read_csv(datasource1, usecols=[1])
    dataframe1 = dataframe1.fillna(method='pad')
    dataset1 = dataframe1.values
    dataset1 = dataset1.astype('float32')
    dataset1 = dataset1[0:sp]

    dataframe2 = pandas.read_csv(datasource2, usecols=[1])
    dataframe2 = dataframe2.fillna(method='pad')
    dataset2 = dataframe2.values
    dataset2 = dataset2.astype('float32')
    dataset2 = dataset2[0:sp]

    dataframe3 = pandas.read_csv(datasource3, usecols=[1])
    dataframe3 = dataframe3.fillna(method='pad')
    dataset3 = dataframe3.values
    dataset3 = dataset3.astype('float32')
    dataset3 = dataset3[0:sp]

    dataframe4 = pandas.read_csv(datasource4, usecols=[1])
    dataframe4 = dataframe4.fillna(method='pad')
    dataset4 = dataframe4.values
    dataset4 = dataset4.astype('float32')

    dataset = numpy.concatenate((dataset1, dataset2, dataset3, dataset4), axis=0)

    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset = scaler.fit_transform(dataset)
    return dataset, scaler

In [5]:
def load_dataset_2(datasource1: str, datasource2: str) -> (numpy.ndarray, MinMaxScaler):
    dataframe1 = pandas.read_csv(datasource1, usecols=[1])
    dataset1 = dataframe1.values.astype('float32')

    dataframe2 = pandas.read_csv(datasource2, usecols=[1])
    dataset2 = dataframe2.values.astype('float32')

    dataset = numpy.concatenate((dataset1, dataset2), axis=0)

    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset = scaler.fit_transform(dataset)

    return dataset, scaler

def load_dataset_3(datasource1: str) -> (numpy.ndarray, MinMaxScaler):
    dataframe1 = pandas.read_csv(datasource1, usecols=[1])
    dataset1 = dataframe1.values.astype('float32')

    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset = scaler.fit_transform(dataset1)

    return dataset, scaler

In [6]:
def create_dataset(dataset: numpy.ndarray, look_back: int=1) -> (numpy.ndarray, numpy.ndarray):
    data_x, data_y = [], []
    for i in range(len(dataset) - look_back):
        a = dataset[i : (i + look_back), 0]
        data_x.append(a)
        data_y.append(dataset[i + look_back, 0])
    return numpy.array(data_x), numpy.array(data_y)

In [ ]:
from keras.models import Model
from keras.layers import RepeatVector, TimeDistributed
from keras.optimizers import Adam
from keras.metrics import RootMeanSquaredError

def build_model(input_shape, blocks_num=84, dropout_rate=1.1e-2, lr=2.6e-3) -> Sequential:
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(ATSLSTM(blocks_num, stateful=False))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1))
    optimizer = optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, amsgrad=False)
    model.compile(loss='mean_squared_error', optimizer=optimizer)
    return model

def build_model_2(input_shape, blocks_num=32, dropout_rate=0.0, lr=0.001) -> Sequential:
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(LSTM(blocks_num, stateful=False))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1))
    optimizer = optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, amsgrad=False)
    model.compile(loss='mean_squared_error', optimizer=optimizer)
    return model

In [8]:
def make_forecast(model: Sequential, look_back_buffer: numpy.ndarray, timesteps: int=1, batch_size: int=1):
    forecast_predict = numpy.empty((0, 1), dtype=numpy.float32)

    for _ in trange(timesteps, desc='predicting data', mininterval=1.0):
        cur_predict = model.predict(look_back_buffer, batch_size=batch_size)
        forecast_predict = numpy.concatenate([forecast_predict, cur_predict], axis=0)

        # Prepare next input
        cur_predict = cur_predict.reshape(1, 1, 1)
        look_back_buffer = numpy.concatenate([look_back_buffer[:, 1:, :], cur_predict], axis=1)

    return forecast_predict

def make_forecast_until_EOL(model, look_back_buffer, scaler, capacity_threshold=1.4, batch_size=1):
    forecast_predict = numpy.empty((0, 1), dtype=numpy.float32)
    step = 0
    predicted_capacity = float('inf')

    while predicted_capacity > capacity_threshold and step < 300:
        cur_predict = model.predict(look_back_buffer, batch_size=batch_size)
        forecast_predict = numpy.concatenate([forecast_predict, cur_predict], axis=0)
        predicted_capacity = scaler.inverse_transform(cur_predict)[0][0]
        step += 1

        # Prepare next input
        cur_predict = cur_predict.reshape(1, 1, 1)
        look_back_buffer = look_back_buffer.reshape(1, LOOK_BACK, 1)
        look_back_buffer = numpy.concatenate([look_back_buffer[:, 1:, :], cur_predict], axis=1)

    return numpy.array(forecast_predict), step

In [9]:
datasource5 = r'./data/rul/5-capacity168.csv'
datasource6 = r'./data/rul/6-capacity168.csv'
datasource7 = r'./data/rul/7-capacity168.csv'
datasource18 = r'./data/rul/18-capacity132.csv'

dataset, scaler = load_dataset_2(datasource5, datasource6)
joblib.dump(scaler, r'./result/scaler_rul.pickle')
test_dataset, test_scaler = load_dataset_3(datasource18)

In [10]:
dataset.shape

(336, 1)

In [11]:
test_dataset.shape

(132, 1)

In [12]:
battery_splits = {
    "B0005": 168,
    "B0018": 168
}

X_seqs, y_seqs = [], []
start = 0
for battery_id, num_cycles in battery_splits.items():
    end = start + num_cycles
    battery_data = dataset[start:end]

    X_battery, y_battery = create_dataset(battery_data, LOOK_BACK)

    X_seqs.append(X_battery)
    y_seqs.append(y_battery)
    start = end

dataset_x = numpy.concatenate(X_seqs, axis=0)
dataset_y = numpy.concatenate(y_seqs, axis=0)

In [13]:
dataset_x = numpy.expand_dims(dataset_x, axis=-1)
dataset_x.shape, dataset_y.shape

((276, 30, 1), (276,))

In [14]:
test_x, test_y = create_dataset(test_dataset, LOOK_BACK)
test_x = numpy.expand_dims(test_x, axis=-1)
test_x.shape, test_y.shape

((102, 30, 1), (102,))

In [15]:
df_b0005 = pandas.read_csv(datasource18, usecols=[1]).fillna(method='pad')
true_b0005 = df_b0005.values.astype('float32').flatten()

true_b0005 = true_b0005[SP:]

C:\Users\HLC\AppData\Local\Temp\ipykernel_3568\3380972214.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_b0005 = pandas.read_csv(datasource18, usecols=[1]).fillna(method='pad')


In [16]:
loss_list = []
starttime = time.time()

model = build_model(input_shape=(LOOK_BACK, 1), blocks_num=BLOCKS_NUM, dropout_rate=DROPOUT_RATE, lr=LR)

for _ in trange(EPOCHS, desc='fitting model\t', mininterval=1.0):
    history = model.fit(dataset_x, dataset_y, epochs=1, batch_size=BATCH_SIZE, verbose=1, shuffle=False)
    plot_model(model, to_file=r'./result/rul_model_structure.png', show_shapes=True)
    loss_list.append(history.history['loss'][0])
    with open('./result/rul_loss.txt', 'a', encoding='utf-8') as f:
        f.write(str(history.history['loss'][0]) + "\n")
    for layer in model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

model.save(r'./result/rul_model.h5')
endtime = time.time()
dtime = endtime - starttime

# generate predictions for training
dataset_predict = model.predict(dataset_x, batch_size=BATCH_SIZE)

look_back_buffer = test_x[SAMPLE_NUM - 1].reshape(1, LOOK_BACK, 1)
forecast_predict, predicted_rul = make_forecast_until_EOL(model, look_back_buffer, scaler=scaler,
                                                          capacity_threshold=1.4, batch_size=BATCH_SIZE)

# invert dataset and predictions
dataset = scaler.inverse_transform(dataset)
dataset_predict = scaler.inverse_transform(dataset_predict)
dataset_y = scaler.inverse_transform([dataset_y])
forecast_predict = scaler.inverse_transform(forecast_predict)

with open(r'./result/rul_prediction_data' + ".txt", 'a', encoding='utf-8') as f:
    for m in range(len(forecast_predict)):
        f.write(str(forecast_predict[m]) + "\n")
print("Training time: %.8s s" % dtime)
index = []

dataset_score = sqrt(mean_squared_error(dataset_y[0], dataset_predict[:, 0]))
print('Train Dataset Score: %.4f RMSE' % dataset_score)
COMPARE_TIMESTEPS = min(TIMESTEPS, predicted_rul)
forecast_score = sqrt(mean_squared_error(true_b0005[:COMPARE_TIMESTEPS], forecast_predict[:COMPARE_TIMESTEPS, 0]))
print('Test Dataset Score: %.4f RMSE' % forecast_score)

true_rul = 96 - SP
ae = abs(predicted_rul - true_rul)
print('Predicted RUL: %d' % predicted_rul)
print('AE: %d' % ae)

index.append('train_dataset_score: %.5f' % dataset_score)
index.append('test_dataset_score: %.5f' % forecast_score)
index.append('rul_error: %d' % ae)
index.append('time: %1f' % dtime)

with open(r'./result/soh_prediction_result_#7_50.txt', 'a', encoding='utf-8') as f:
    for j in range(len(index)):
        f.write(str(index[j]) + "\n")

fitting model	:   0%|          | 0/170 [00:00<?, ?it/s]c:\Users\HLC\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:731: UserWarning: Gradients do not exist for variables ['kernel', 'recurrent_kernel', 'bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.2460


fitting model	:   1%|          | 1/170 [00:03<10:02,  3.57s/it]

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1935
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1519


fitting model	:   2%|▏         | 3/170 [00:04<03:38,  1.31s/it]

...

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 4.4887e-04
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 4.9742e-04
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 4.4488e-04


fitting model	:  98%|█████████▊| 167/170 [01:26<00:01,  2.05it/s]

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 4.3785e-04
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 5.1388e-04
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.6341e-04


fitting model	: 100%|██████████| 170/170 [01:27<00:00,  1.94it/s]


17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Training time: 87.71831 s
Train Dataset Score: 0.0201 RMSE
Test Dataset Score: 0.1390 RMSE
Predicted RUL: 1
AE: 5
